# Краткое описание
- Цель эксперимента - обучить и сравнить модели детоксикации текста.
- Данные - ru_paradetox (train/val/test).
- Основные выводы - выбрана лучшая модель по качеству и сохранены артефакты и конфиги.


In [ ]:

from pathlib import Path
import json
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

if "y_true" in globals():
    _y_true = y_true
elif "y_test" in globals():
    _y_true = y_test
else:
    raise ValueError("y_true or y_test not found")

if "y_pred" in globals():
    _y_pred = y_pred
elif "y_pred_labels" in globals():
    _y_pred = y_pred_labels
elif "best_model_preds" in globals():
    _y_pred = best_model_preds
else:
    raise ValueError("y_pred not found")

if "y_proba" in globals():
    _y_score = y_proba
elif "best_model_probs" in globals():
    _y_score = best_model_probs
else:
    _y_score = _y_pred

_y_true_arr = np.array(_y_true)
_avg = "micro" if _y_true_arr.ndim > 1 and _y_true_arr.shape[1] > 1 else "binary"

metrics = {
    "accuracy": float(accuracy_score(_y_true, _y_pred)),
    "precision": float(precision_score(_y_true, _y_pred, average=_avg, zero_division=0)),
    "recall": float(recall_score(_y_true, _y_pred, average=_avg, zero_division=0)),
    "f1": float(f1_score(_y_true, _y_pred, average=_avg, zero_division=0)),
}

try:
    metrics["roc_auc"] = float(roc_auc_score(_y_true, _y_score))
except Exception:
    metrics["roc_auc"] = float(roc_auc_score(_y_true, _y_score, average=_avg, multi_class="ovr"))

configs_dir = Path("configs_dir")
configs_dir.mkdir(parents=True, exist_ok=True)
with open(configs_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved metrics to", configs_dir / "metrics.json")
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
from functools import partial
import random


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm.auto import tqdm


import joblib
import pickle


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, confusion_matrix
)


from scipy import sparse


try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
except Exception:
    MultilabelStratifiedKFold = None
    MultilabelStratifiedShuffleSplit = None


try:

    from gensim.models import KeyedVectors
except Exception:
    KeyedVectors = None

try:
    import fasttext
    import fasttext.util
except Exception:
    fasttext = None


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
except Exception:
    torch = None


try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception:
    AutoTokenizer = AutoModelForSequenceClassification = TrainingArguments = Trainer = DataCollatorWithPadding = None


import warnings
warnings.filterwarnings("ignore")


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [2]:
def set_seed(seed: int = 42) -> None:
    # Фиксируем seed для воспроизводимости (насколько это возможно).
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


    global DATA_LOADER_GEN
    try:
        DATA_LOADER_GEN = torch.Generator()
        DATA_LOADER_GEN.manual_seed(seed)
    except Exception:
        DATA_LOADER_GEN = None

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [3]:


# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.tsv', 'validation': 'dev.tsv'}
df_train = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["train"], sep="\t")
df_val = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["validation"], sep="\t")

In [4]:
df_train.head()

,ru_toxic_comment,ru_neutral_comment
0,"и,чё,блядь где этот херой был до этого со свои...","Ну и где этот герой был,со своими доказательст..."
1,"и,чё,блядь где этот херой был до этого со свои...",Где этот герой был до этого со своими доказате...
2,"и,чё,блядь где этот херой был до этого со свои...","и,где этот герой был до этого со своими доказа..."
3,"О, а есть деанон этого петуха?","О, а есть деанон"
4,"херну всякую пишут,из-за этого лайка.долбоебизм.","Чушь всякую пишут, из- за этого лайка."


In [5]:
df_val.head()

,ru_toxic_comment,ru_neutral_comment
0,пиздеж! температуры горения хватит чтобы её ра...,Враньё! Температуры горения хватит чтобы ее ра...
1,пиздеж! температуры горения хватит чтобы её ра...,"неправда,температуры горения хватит чтобы расп..."
2,пиздеж! температуры горения хватит чтобы её ра...,Враньё! Температуры горения хватит на чтобы её...
3,а ты чмо там был.ты вообще служил.гандон,А ты там был? Ты вообще служил?
4,пиздабол ---- а сам где кормишься ?,а сам где кормишься ?


In [6]:
df_train.shape

(11090, 2)

In [7]:
df_val.shape

(1116, 2)

In [8]:
df_train = df_train.drop_duplicates().reset_index(drop=True)
df_val = df_val.drop_duplicates().reset_index(drop=True)

mask_error_train = df_train.astype(str).apply(lambda col: col.str.contains(r"#ERROR!", na=False)).any(axis=1)
mask_error_val = df_val.astype(str).apply(lambda col: col.str.contains(r"#ERROR!", na=False)).any(axis=1)

df_train = df_train.loc[~mask_error_train].reset_index(drop=True)
df_val = df_val.loc[~mask_error_val].reset_index(drop=True)


In [9]:
df_train.shape

(10882, 2)

In [10]:
df_val.shape

(1087, 2)

In [11]:
df_train = df_train.sample(frac=0.2, random_state=42).reset_index(drop=True)
df_val = df_val.sample(frac=0.2, random_state=42).reset_index(drop=True)


In [12]:
df_train, df_test = train_test_split(df_train, test_size=0.1, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [13]:
max_len = 100
vocab_size = 20000
special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

def tokenize(text: str):
    return re.findall(r"\w+|[^\w\s]", str(text).lower(), re.UNICODE)

all_texts = pd.concat([
    df_train["ru_toxic_comment"].astype(str),
    df_train["ru_neutral_comment"].astype(str)
], ignore_index=True)

counter = Counter()
for text in all_texts:
    counter.update(tokenize(text))

most_common = [w for w, _ in counter.most_common(vocab_size - len(special_tokens))]
idx2word = special_tokens + most_common
word2idx = {w: i for i, w in enumerate(idx2word)}

pad_id = word2idx["<PAD>"]
bos_id = word2idx["<BOS>"]
eos_id = word2idx["<EOS>"]
unk_id = word2idx["<UNK>"]


In [14]:
def encode_text(text, max_len):
    toks = tokenize(text)
    tokens = [word2idx.get(w, unk_id) for w in toks]
    tokens = [bos_id] + tokens[:max_len - 2] + [eos_id]
    if len(tokens) < max_len:
        tokens = tokens + [pad_id] * (max_len - len(tokens))
    return tokens

def make_arrays(df):
    src = np.array([encode_text(t, max_len) for t in df["ru_toxic_comment"].astype(str)])
    tgt = np.array([encode_text(t, max_len) for t in df["ru_neutral_comment"].astype(str)])
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    return src, tgt_in, tgt_out


In [15]:
class Seq2SeqDataset(Dataset):
    def __init__(self, src, tgt_in, tgt_out):
        self.src = torch.LongTensor(src)
        self.tgt_in = torch.LongTensor(tgt_in)
        self.tgt_out = torch.LongTensor(tgt_out)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt_in[idx], self.tgt_out[idx]


In [16]:
X_train_src, X_train_tgt_in, X_train_tgt_out = make_arrays(df_train)
X_val_src, X_val_tgt_in, X_val_tgt_out = make_arrays(df_val)
X_test_src, X_test_tgt_in, X_test_tgt_out = make_arrays(df_test)

# Diagnostics: how many UNK in targets
def unk_stats(arr, name):
    total = arr.size
    unk_count = (arr == unk_id).sum()
    print(f"{name}: total tokens={total}, UNK tokens={unk_count}, UNK%={unk_count/total*100:.2f}")

unk_stats(X_train_tgt_out, 'train targets')
unk_stats(X_val_tgt_out, 'val targets')


train targets: total tokens=193842, UNK tokens=0, UNK%=0.00
val targets: total tokens=21483, UNK tokens=543, UNK%=2.53


In [17]:
class RNNSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.RNN(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        self.decoder = nn.RNN(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.bidirectional = bidirectional
        enc_out_dim = hidden_size * 2 if bidirectional else hidden_size
        self.enc_proj = nn.Linear(enc_out_dim, hidden_size) if bidirectional else nn.Identity()
        self.attn_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def _merge_directions(self, h):
        if not self.bidirectional:
            return h
        h = h.view(self.num_layers, 2, h.size(1), self.hidden_size)
        return h.sum(dim=1)

    def encode(self, src):
        src_emb = self.dropout(self.embedding(src))
        enc_out, h = self.encoder(src_emb)
        enc_out = self.enc_proj(enc_out)
        h = self._merge_directions(h)
        return enc_out, h

    def decode_step(self, token, hidden, enc_out):
        emb = self.dropout(self.embedding(token))
        dec_out, hidden = self.decoder(emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits, hidden

    def forward(self, src, tgt_in):
        enc_out, h = self.encode(src)
        tgt_emb = self.dropout(self.embedding(tgt_in))
        dec_out, _ = self.decoder(tgt_emb, h)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits

class GRUSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.GRU(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        self.decoder = nn.GRU(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                              dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.bidirectional = bidirectional
        enc_out_dim = hidden_size * 2 if bidirectional else hidden_size
        self.enc_proj = nn.Linear(enc_out_dim, hidden_size) if bidirectional else nn.Identity()
        self.attn_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def _merge_directions(self, h):
        if not self.bidirectional:
            return h
        h = h.view(self.num_layers, 2, h.size(1), self.hidden_size)
        return h.sum(dim=1)

    def encode(self, src):
        src_emb = self.dropout(self.embedding(src))
        enc_out, h = self.encoder(src_emb)
        enc_out = self.enc_proj(enc_out)
        h = self._merge_directions(h)
        return enc_out, h

    def decode_step(self, token, hidden, enc_out):
        emb = self.dropout(self.embedding(token))
        dec_out, hidden = self.decoder(emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits, hidden

    def forward(self, src, tgt_in):
        enc_out, h = self.encode(src)
        tgt_emb = self.dropout(self.embedding(tgt_in))
        dec_out, _ = self.decoder(tgt_emb, h)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits

class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.LSTM(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                               dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        self.decoder = nn.LSTM(embed_dim, hidden_size, batch_first=True, num_layers=num_layers,
                               dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.bidirectional = bidirectional
        enc_out_dim = hidden_size * 2 if bidirectional else hidden_size
        self.enc_proj = nn.Linear(enc_out_dim, hidden_size) if bidirectional else nn.Identity()
        self.attn_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def _merge_directions(self, state):
        if not self.bidirectional:
            return state
        h, c = state
        h = h.view(self.num_layers, 2, h.size(1), self.hidden_size).sum(dim=1)
        c = c.view(self.num_layers, 2, c.size(1), self.hidden_size).sum(dim=1)
        return h, c

    def encode(self, src):
        src_emb = self.dropout(self.embedding(src))
        enc_out, (h, c) = self.encoder(src_emb)
        enc_out = self.enc_proj(enc_out)
        h, c = self._merge_directions((h, c))
        return enc_out, (h, c)

    def decode_step(self, token, hidden, enc_out):
        emb = self.dropout(self.embedding(token))
        dec_out, hidden = self.decoder(emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits, hidden

    def forward(self, src, tgt_in):
        enc_out, hidden = self.encode(src)
        tgt_emb = self.dropout(self.embedding(tgt_in))
        dec_out, _ = self.decoder(tgt_emb, hidden)
        attn_scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        attn_input = torch.cat([dec_out, context], dim=-1)
        attn_out = torch.tanh(self.attn_proj(attn_input))
        logits = self.output(attn_out)
        return logits


In [18]:
def train_seq2seq(model, train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32, weight_decay=1e-4, patience=3):
    model = model.to(device)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, generator=DATA_LOADER_GEN)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1)
    best_val = float('inf')
    best_state = None
    bad_epochs = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, tgt_in, tgt_out in train_loader:
            src = src.to(device)
            tgt_in = tgt_in.to(device)
            tgt_out = tgt_out.to(device)
            optimizer.zero_grad()
            logits = model(src, tgt_in)
            loss = loss_fn(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for src, tgt_in, tgt_out in val_loader:
                src = src.to(device)
                tgt_in = tgt_in.to(device)
                tgt_out = tgt_out.to(device)
                logits = model(src, tgt_in)
                loss = loss_fn(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
                val_loss += loss.item()
        val_avg = val_loss / len(val_loader)
        print(f"epoch {epoch+1} train_loss {total_loss/len(train_loader):.4f} val_loss {val_avg:.4f}")
        if val_avg < best_val:
            best_val = val_avg
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [19]:
def greedy_decode(model, src, max_len):
    model.eval()
    src = src.to(device)
    enc_out, hidden = model.encode(src)
    decoded = torch.full((src.size(0), 1), bos_id, dtype=torch.long, device=src.device)
    for _ in range(max_len - 1):
        last = decoded[:, -1:].contiguous()
        logits, hidden = model.decode_step(last, hidden, enc_out)
        next_token = logits.argmax(dim=-1)
        decoded = torch.cat([decoded, next_token], dim=1)
    return decoded


In [20]:
def decode_sequences(seqs):
    texts = []
    for seq in seqs:
        words = []
        for idx in seq:
            if idx == eos_id:
                break
            if idx in (pad_id, bos_id):
                continue
            words.append(idx2word[idx] if idx < len(idx2word) else "")
        texts.append(" ".join(words).strip())
    return texts


In [21]:
train_ds = Seq2SeqDataset(X_train_src, X_train_tgt_in, X_train_tgt_out)
val_ds = Seq2SeqDataset(X_val_src, X_val_tgt_in, X_val_tgt_out)


In [22]:
# train_ds[:5]

In [23]:
# val_ds[:5]

In [24]:
model_rnn = train_seq2seq(RNNSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32)



epoch 1 train_loss 7.9818 val_loss 7.2081
epoch 2 train_loss 6.8059 val_loss 7.1169
epoch 3 train_loss 6.7103 val_loss 7.1235
epoch 4 train_loss 6.6752 val_loss 7.1307
epoch 5 train_loss 6.6303 val_loss 7.1126
epoch 6 train_loss 6.5940 val_loss 7.0945
epoch 7 train_loss 6.5421 val_loss 7.0631
epoch 8 train_loss 6.5080 val_loss 7.0462
epoch 9 train_loss 6.4662 val_loss 7.0071
epoch 10 train_loss 6.4214 val_loss 6.9892


In [25]:
model_gru = train_seq2seq(GRUSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32)


epoch 1 train_loss 7.9734 val_loss 7.3091
epoch 2 train_loss 6.7901 val_loss 7.2398
epoch 3 train_loss 6.7272 val_loss 7.2397
epoch 4 train_loss 6.7100 val_loss 7.2300
epoch 5 train_loss 6.6649 val_loss 7.2027
epoch 6 train_loss 6.6187 val_loss 7.1437
epoch 7 train_loss 6.5594 val_loss 7.1088
epoch 8 train_loss 6.4936 val_loss 7.0874
epoch 9 train_loss 6.4330 val_loss 7.0318
epoch 10 train_loss 6.3734 val_loss 6.9663


In [26]:
model_lstm = train_seq2seq(LSTMSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=10, lr=3e-4, batch_size=32)

epoch 1 train_loss 8.0227 val_loss 7.2552
epoch 2 train_loss 6.7917 val_loss 7.2217
epoch 3 train_loss 6.7171 val_loss 7.1991
epoch 4 train_loss 6.6870 val_loss 7.1675
epoch 5 train_loss 6.6543 val_loss 7.1849
epoch 6 train_loss 6.6204 val_loss 7.1529
epoch 7 train_loss 6.5787 val_loss 7.1598
epoch 8 train_loss 6.5256 val_loss 7.0756
epoch 9 train_loss 6.4834 val_loss 7.0448
epoch 10 train_loss 6.4337 val_loss 7.0200


In [27]:
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

from tqdm.auto import tqdm

modelname = "cointegrated/rut5-small"
maxlen = 128

batchsize = 8
numepochs = 2
lr = 1e-5
logsteps = 50
seed = 42

torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(modelname, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(modelname).to(device)

def cleandf(df):
    df = df.dropna()
    df = df[df["ru_toxic_comment"].str.len() > 0]
    df = df[df["ru_neutral_comment"].str.len() > 0]
    return df

df_train = cleandf(df_train)
df_val = cleandf(df_val)

class DetoxSeq2SeqDataset(Dataset):
    def __init__(self, df, tokenizer, maxlen):
        self.inputs = tokenizer(
            ["детоксифицируй текст: " + str(x) for x in df["ru_toxic_comment"].tolist()],
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )

        labels = tokenizer(
            df["ru_neutral_comment"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )["input_ids"]

        labels[labels == tokenizer.pad_token_id] = -100
        self.labels = labels

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.inputs.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = DetoxSeq2SeqDataset(df_train, tokenizer, maxlen)
val_ds = DetoxSeq2SeqDataset(df_val, tokenizer, maxlen)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

train_loader = DataLoader(train_ds, batch_size=batchsize, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=batchsize, shuffle=False, collate_fn=collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

num_training_steps = numepochs * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

def eval_loss(model, loader):
    model.eval()
    total = 0.0
    n = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss

            if torch.isnan(loss):
                continue

            total += loss.item()
            n += 1

    return total / max(1, n)

global_step = 0

for epoch in range(numepochs):
    model.train()

    for step, batch in enumerate(tqdm(train_loader, desc=f"train {epoch+1}/{numepochs}"), start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        out = model(**batch)
        loss = out.loss

        if torch.isnan(loss):
            continue

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        global_step += 1

        if global_step % logsteps == 0:
            print(f"step={global_step} loss={loss.item():.4f}")

    val_loss = eval_loss(model, val_loader)
    print(f"epoch={epoch+1} val_loss={val_loss:.4f}")


class DetoxGenDataset(Dataset):
    def __init__(self, df, tokenizer, maxlen):
        self.inputs = tokenizer(
            ["детоксифицируй текст: " + str(x) for x in df["ru_toxic_comment"].tolist()],
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )

    def __len__(self):
        return self.inputs["input_ids"].size(0)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.inputs.items()}

gen_ds = DetoxGenDataset(df_val, tokenizer, maxlen)
gen_loader = DataLoader(gen_ds, batch_size=batchsize, shuffle=False)

model.eval()
preds = []

with torch.no_grad():
    for batch in tqdm(gen_loader, desc="generate"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            num_beams=4,
        )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        preds.extend([x.strip() for x in decoded])

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
train 1/2:  20%|██        | 50/245 [00:33<02:28,  1.32it/s]

step=50 loss=2.1418


train 1/2:  41%|████      | 100/245 [01:08<01:52,  1.29it/s]

step=100 loss=2.7312


train 1/2:  61%|██████    | 150/245 [01:44<01:12,  1.32it/s]

...

train 2/2:  84%|████████▎ | 205/245 [03:17<00:43,  1.09s/it]

step=450 loss=2.9374


train 2/2: 100%|██████████| 245/245 [04:00<00:00,  1.02it/s]


epoch=2 val_loss=1.7407


generate: 100%|██████████| 28/28 [01:48<00:00,  3.87s/it]


In [28]:
# model.eval()
#
# examples = []
#
# with torch.no_grad():
#     for i, batch in enumerate(tqdm(gen_loader, desc="generate")):
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#
#         generated = model.generate(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             max_new_tokens=64,
#             num_beams=4,
#         )
#
#         decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
#
#         for j in range(len(decoded)):
#             idx = i * batchsize + j
#             if idx >= len(df_val):
#                 break
#
#             examples.append({
#                 "toxic": df_val.iloc[idx]["ru_toxic_comment"],
#                 "pred": decoded[j].strip(),
#                 "true": df_val.iloc[idx]["ru_neutral_comment"],
#             })
#
# df_show = pd.DataFrame(examples)
#
# df_show.head(20)

In [29]:
val_src_tensor = torch.LongTensor(X_val_src)

def beam_search_decode(model, src, max_len, beam_width=4):
    model.eval()
    device = next(model.parameters()).device
    outputs = []
    with torch.no_grad():
        for i in range(src.size(0)):
            s = src[i:i+1].to(device)
            enc_out, init_hidden = model.encode(s)

            # beam: list of (tokens, hidden, score)
            beams = [([bos_id], init_hidden, 0.0)]
            completed = []
            for _ in range(max_len - 1):
                new_beams = []
                for tokens, hidden, score in beams:
                    last = torch.LongTensor([[tokens[-1]]]).to(device)
                    logits, h_next = model.decode_step(last, hidden, enc_out)
                    logits = logits[:, -1, :]
                    logp = torch.log_softmax(logits, dim=-1).squeeze(0)
                    topk = torch.topk(logp, beam_width)
                    for k in range(beam_width):
                        token = int(topk.indices[k].item())
                        token_score = float(topk.values[k].item())
                        new_tokens = tokens + [token]
                        new_score = score + token_score
                        new_beams.append((new_tokens, h_next, new_score))
                # keep top beams
                new_beams = sorted(new_beams, key=lambda x: x[2], reverse=True)[:beam_width]
                beams = []
                for tokens, hidden, score in new_beams:
                    if tokens[-1] == eos_id:
                        completed.append((tokens, score))
                    else:
                        beams.append((tokens, hidden, score))
                if len(beams) == 0:
                    break
            if len(completed) == 0:
                completed = [(b[0], b[2]) for b in beams]
            best = sorted(completed, key=lambda x: x[1], reverse=True)[0][0]
            outputs.append(best)
    # pad/truncate outputs to same length
    max_out_len = max(len(x) for x in outputs)
    padded = [x + [pad_id] * (max_out_len - len(x)) for x in outputs]
    return torch.LongTensor(padded)

preds_rnn = decode_sequences(beam_search_decode(model_rnn, val_src_tensor, max_len, beam_width=4).numpy())
preds_gru = decode_sequences(beam_search_decode(model_gru, val_src_tensor, max_len, beam_width=4).numpy())
preds_lstm = decode_sequences(beam_search_decode(model_lstm, val_src_tensor, max_len, beam_width=4).numpy())

refs = df_val["ru_neutral_comment"].astype(str).tolist()


In [30]:
try:
    import evaluate
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")
except Exception as e:
    raise ImportError("Install evaluate to compute BLEU/ROUGE/BERTScore") from e


In [31]:
results = []
preds_transformer = preds
refs_for_bleu = [[r] for r in refs]
for name, preds in [("RNN", preds_rnn), ("GRU", preds_gru), ("LSTM", preds_lstm), ("Transformer", preds_transformer)]:
    try:
        bleu_score = bleu.compute(predictions=preds, references=refs_for_bleu)["bleu"]
    except Exception:
        bleu_score = 0.0
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="ru")
    results.append({
        "model": name,
        "bleu": bleu_score,
        "rouge1": rouge_scores.get("rouge1"),
        "rouge2": rouge_scores.get("rouge2"),
        "rougeL": rouge_scores.get("rougeL"),
        "bertscore_f1": float(np.mean(bert_scores["f1"]))
    })

metrics_df = pd.DataFrame(results)
print(metrics_df)


         model      bleu    rouge1    rouge2    rougeL  bertscore_f1
0          RNN  0.000000  0.000000  0.000000  0.000000      0.539521
1          GRU  0.000000  0.000000  0.000000  0.000000      0.595944
2         LSTM  0.000000  0.000000  0.000000  0.000000      0.615322
3  Transformer  0.407831  0.036866  0.009217  0.036866      0.831882


In [ ]:

metrics_df = metrics_df.sort_values(by="bertscore_f1", ascending=False)

best_name = metrics_df.iloc[0]["model"]

if best_name == "RNN":
    best_model = model_rnn
elif best_name == "GRU":
    best_model = model_gru
elif best_name == "LSTM":
    best_model = model_lstm
elif best_name == "Transformer":
    best_model = model

# Сохраняем state_dict лучшей модели
artifacts_dir = Path("../../artifacts/detox_model").resolve()
artifacts_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_model.state_dict(), str(artifacts_dir / "best_detox_model.pt"))

# Сохраняем конфиги (hyperparams, train params, inference params)
config_dir = Path("../../configs/detox_model").resolve()
config_dir.mkdir(parents=True, exist_ok=True)

hyperparams = {}
hyperparams["model_type"] = best_name
try:
    emb = best_model.embedding
    hyperparams["vocab_size"] = int(getattr(emb, "num_embeddings", None) or len(idx2word))
    hyperparams["embed_dim"] = int(getattr(emb, "embedding_dim", None))
except Exception:
    hyperparams["vocab_size"] = len(idx2word)

# common attributes
hyperparams["hidden_size"] = int(getattr(best_model, "hidden_size", None) or 128)
hyperparams["num_layers"] = int(getattr(best_model, "num_layers", None) or 2)
hyperparams["dropout"] = float(getattr(best_model, "dropout", nn.Dropout(0.3)).p if hasattr(getattr(best_model, "dropout", None), 'p') else 0.3)
hyperparams["bidirectional"] = bool(getattr(best_model, "bidirectional", False))

train_params = {
    "epochs": 10,
    "lr": 3e-4,
    "batch_size": 32,
    "weight_decay": 1e-4,
    "patience": 3,
    "optimizer": "AdamW",
}

inference_params = {
    "max_len": int(max_len) if 'max_len' in globals() else 100,
    "beam_width": 4,
    "bos_id": int(bos_id),
    "eos_id": int(eos_id),
    "pad_id": int(pad_id),
    "unk_id": int(unk_id),
}

import json

with open(config_dir / "hyperparams.json", "w", encoding="utf-8") as f:
    json.dump(hyperparams, f, ensure_ascii=False, indent=2)

with open(config_dir / "train_params.json", "w", encoding="utf-8") as f:
    json.dump(train_params, f, ensure_ascii=False, indent=2)

with open(config_dir / "inference_params.json", "w", encoding="utf-8") as f:
    json.dump(inference_params, f, ensure_ascii=False, indent=2)

# Сохраняем метрики в config_dir/metrics.json
try:
    metrics_out = metrics_df.where(pd.notnull(metrics_df), None).to_dict(orient="records")
except Exception:
    try:
        metrics_out = metrics_df.to_dict(orient="records")
    except Exception:
        metrics_out = []

with open(config_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, ensure_ascii=False, indent=2)

print("Saved metrics to", config_dir / "metrics.json")